# Model Training Notebook 01: Baseline Forecasting Models
**Standalone Training, Walk-Forward Validation & Serialization**
- **Models Trained**:
  1. `NaivePersistenceModel` ($\hat{y}_{t+h} = y_t$)
  2. `MovingAverageModel` (Window-tuned $k \in \{3, 7, 14, 21\}$ Rolling Mean)
- **Targets**: `hhs_care` (Children in HHS Care) & `hhs_discharged` (Children Discharged from HHS Care)
- **Saved Artifacts**: `models/baseline_naive_model.pkl`, `models/baseline_ma7_model.pkl`

In [1]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

CSV_PATH = "../data/HHS_Unaccompanied_Alien_Children_Program.csv", "data/HHS_Unaccompanied_Alien_Children_Program.csv", "HHS_Unaccompanied_Alien_Children_Program.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "user_uploaded_files/ttl=63d/user_uploaded/HHS_Unaccompanied_Alien_Children_Program.csv"

# 1. Load Raw CSV & Clean Trailing Empty Rows / Comma Formatting
raw_df = pd.read_csv(CSV_PATH).dropna(how="all").copy()
raw_df.columns = [c.strip() for c in raw_df.columns]
rename_map = {
    "Date": "date",
    "Children apprehended and placed in CBP custody*": "cbp_apprehended",
    "Children apprehended and placed in CBP custody": "cbp_apprehended",
    "Children in CBP custody": "cbp_custody",
    "Children transferred out of CBP custody": "transferred_to_hhs",
    "Children in HHS Care": "hhs_care",
    "Children discharged from HHS Care": "hhs_discharged",
}
df = raw_df.rename(columns=rename_map)
df["date"] = pd.to_datetime(df["date"], format="mixed")

num_cols = ["cbp_apprehended", "cbp_custody", "transferred_to_hhs", "hhs_care", "hhs_discharged"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

# 2. Sort Chronologically & Reindex to Strict Daily Frequency (1,075 Continuous Days)
df = df.sort_values("date").drop_duplicates(subset=["date"], keep="last").set_index("date")
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
df["is_observed"] = 1
df = df.reindex(full_idx)
df.index.name = "date"
df["is_observed"] = df["is_observed"].fillna(0).astype(int)

# 3. Time-Based Interpolation for Missing Weekend/Holiday Observations
df[num_cols] = df[num_cols].interpolate(method="time").bfill().ffill()
print(f"Dataset Prepared: {df.index.min().date()} to {df.index.max().date()} | Total Days: {len(df)} | Observed: {df['is_observed'].sum()}")

Dataset Prepared: 2023-01-12 to 2025-12-21 | Total Days: 1075 | Observed: 720

In [2]:
# Engineer 72 Predictive Time-Series & Queueing Features
feat = df.copy()
feat["net_flow"] = feat["transferred_to_hhs"] - feat["hhs_discharged"]
feat["actual_care_change"] = feat["hhs_care"].diff().fillna(0)
feat["intake_to_exit_ratio"] = (feat["transferred_to_hhs"] + 1.0) / (feat["hhs_discharged"] + 1.0)
feat["cbp_transfer_efficiency"] = (feat["transferred_to_hhs"] + 1.0) / (feat["cbp_custody"] + 1.0)

# Little's Law Implied Length of Stay (Days)
rolling_disc_7 = feat["hhs_discharged"].rolling(7, min_periods=1).mean().clip(lower=1.0)
feat["implied_los_days"] = feat["hhs_care"] / rolling_disc_7

# Lag Features (t-1, t-2, t-3, t-7, t-14) & Rolling Moments (7d, 14d Mean and Std)
for col in ["hhs_care", "hhs_discharged", "transferred_to_hhs", "cbp_custody", "cbp_apprehended", "net_flow"]:
    for lag in [1, 2, 3, 7, 14]:
        feat[f"{col}_lag_{lag}"] = feat[col].shift(lag)
    for win in [7, 14]:
        feat[f"{col}_roll_mean_{win}"] = feat[col].shift(1).rolling(win).mean()
        feat[f"{col}_roll_std_{win}"] = feat[col].shift(1).rolling(win).std()

feat["care_momentum_7d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(8)
feat["care_momentum_14d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(15)
feat["dayofweek"] = feat.index.dayofweek
feat["month"] = feat.index.month
feat["is_weekend"] = (feat["dayofweek"] >= 5).astype(int)
feat["sin_dow"] = np.sin(2 * np.pi * feat["dayofweek"] / 7.0)
feat["cos_dow"] = np.cos(2 * np.pi * feat["dayofweek"] / 7.0)
feat["sin_month"] = np.sin(2 * np.pi * feat["month"] / 12.0)
feat["cos_month"] = np.cos(2 * np.pi * feat["month"] / 12.0)

feat = feat.dropna()

# Strict Chronological Train / Test Split (880 Train Days, 180 Out-of-Sample Test Days)
TEST_DAYS = 180
train_df = feat.iloc[:-TEST_DAYS].copy()
test_df = feat.iloc[-TEST_DAYS:].copy()
print(f"Feature Matrix: {feat.shape} | Train: {train_df.shape} ({train_df.index.min().date()} to {train_df.index.max().date()}) | Test: {test_df.shape} ({test_df.index.min().date()} to {test_df.index.max().date()})")

Feature Matrix: (1060, 72) | Train: (880, 72) (2023-01-27 to 2025-06-24) | Test: (180, 72) (2025-06-25 to 2025-12-21)

In [3]:
# Define Standalone Baseline Estimators & Evaluation Function
class NaivePersistenceForecaster:
    def __init__(self, target_col: str):
        self.target_col = target_col
        self.last_value_ = None
        self.residual_std_ = None

    def fit(self, df_train: pd.DataFrame):
        self.last_value_ = float(df_train[self.target_col].iloc[-1])
        diffs = df_train[self.target_col].diff().dropna()
        self.residual_std_ = float(diffs.std())
        return self

    def predict_walk_forward(self, df_eval: pd.DataFrame) -> np.ndarray:
        return df_eval[f"{self.target_col}_lag_1"].values

    def forecast_horizon(self, horizon: int) -> np.ndarray:
        return np.full(horizon, self.last_value_)


class MovingAverageForecaster:
    def __init__(self, target_col: str, window: int = 7):
        self.target_col = target_col
        self.window = window
        self.history_tail_ = None

    def fit(self, df_train: pd.DataFrame):
        self.history_tail_ = df_train[self.target_col].iloc[-self.window:].values.copy()
        return self

    def predict_walk_forward(self, df_eval: pd.DataFrame) -> np.ndarray:
        return df_eval[f"{self.target_col}_roll_mean_{self.window}"].values

    def forecast_horizon(self, horizon: int) -> np.ndarray:
        buf = list(self.history_tail_)
        preds = []
        for _ in range(horizon):
            nxt = float(np.mean(buf[-self.window:]))
            preds.append(nxt)
            buf.append(nxt)
        return np.array(preds)


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, model_name: str) -> dict:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mape = float(np.mean(np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + 1.0)) * 100.0)
    acc = max(0.0, 100.0 - mape)
    return {"Model": model_name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE (%)": round(mape, 2), "Accuracy (%)": round(acc, 2)}

# Train & Evaluate on Both Targets
naive_care = NaivePersistenceForecaster("hhs_care").fit(train_df)
ma7_care = MovingAverageForecaster("hhs_care", window=7).fit(train_df)
ma14_care = MovingAverageForecaster("hhs_care", window=14).fit(train_df)

naive_disc = NaivePersistenceForecaster("hhs_discharged").fit(train_df)
ma7_disc = MovingAverageForecaster("hhs_discharged", window=7).fit(train_df)

care_metrics = pd.DataFrame([
    evaluate_predictions(test_df["hhs_care"].values, naive_care.predict_walk_forward(test_df), "Naive Persistence (Care)"),
    evaluate_predictions(test_df["hhs_care"].values, ma7_care.predict_walk_forward(test_df), "Moving Average 7d (Care)"),
    evaluate_predictions(test_df["hhs_care"].values, ma14_care.predict_walk_forward(test_df), "Moving Average 14d (Care)"),
    evaluate_predictions(test_df["hhs_discharged"].values, naive_disc.predict_walk_forward(test_df), "Naive Persistence (Discharge)"),
    evaluate_predictions(test_df["hhs_discharged"].values, ma7_disc.predict_walk_forward(test_df), "Moving Average 7d (Discharge)"),
])
print(care_metrics.to_string(index=False))

# Save Trained Baseline Artifacts
with open("models/baseline_naive_model.pkl", "wb") as f:
    pickle.dump({"care": vars(naive_care), "discharge": vars(naive_disc)}, f)
with open("models/baseline_ma7_model.pkl", "wb") as f:
    pickle.dump({"care": vars(ma7_care), "discharge": vars(ma7_disc)}, f)
print("\nSaved trained baseline models to models/baseline_naive_model.pkl and models/baseline_ma7_model.pkl")

Model   MAE  RMSE  MAPE (%)  Accuracy (%)
     Naive Persistence (Care)  6.93  8.56      0.31         99.69
     Moving Average 7d (Care) 23.23 28.21      1.04         98.96
    Moving Average 14d (Care) 44.18 52.64      1.98         98.02
Naive Persistence (Discharge)  3.96  5.25     31.43         68.57
Moving Average 7d (Discharge)  4.05  5.32     31.51         68.49

Saved trained baseline models to models/baseline_naive_model.pkl and models/baseline_ma7_model.pkl